# 39 · Is human PT axial zonation globally weaker than mouse, and why?

Notebook 37 found that human S3 − early gradients are a small fraction of the mouse ones in our
data and in two external human references. That finding could become the paper's central
biological result. This notebook tests it properly before any claim is made.

The protocol below was written before any result in this notebook was computed (workstream 2,
phase 3). Segment labels, not the reconstructed coordinate, define every contrast, so the results
do not depend on workstream 1's choice of coordinate.

## Pre-registered protocol

**Question.** Is human PT axial zonation globally weaker than mouse? If so, is that biology, or
statistics, sampled region, or tissue state?

**1 · Statistics: a dilution-free amplitude ratio.**
- Regressing our human gradient on our own mouse gradient puts mouse noise in the predictor, so
  the slope shrinks and human looks flat (regression dilution).
- Instead, we regress **both** our human and our mouse gradients on an **independent reference
  gradient**, across genes chosen in that reference. The ratio of the two slopes,
  cov(human, ref) / cov(mouse, ref), is free of reference noise.
- **References:** mouse microdissection (Chen 2021, replicate mice; Chen 2023, both sexes), mouse
  snRNA (Census, per donor, by sex) and human snRNA (Lake et al. 2023 / KPMP, healthy cortex
  donors).
- **Expected bias.** A mouse reference biases the ratio **down** when zonation differs between
  species. A human reference biases it **up**. Together they bracket the answer.
- **Uncertainty.** Two-stage bootstrap: resample reference donors where they exist, then genes,
  2,000 draws. We also report the four human-section × mouse-specimen pairings.
- **Errors-in-variables versions.** Deming slopes, with error-variance ratios from replicate
  variance, and orthogonal regression.
- **Naive comparison.** Our human regressed on our mouse gradient, to show how much dilution
  costs.

**2 · Region.**
- Primary contrast: **S2 − S1**, cortical in both species.
- Secondary contrast: **S3 − early (S1 + S2)**, with mouse S3 restricted to cortical-like
  structures. These are no deeper than the 95th percentile of the same specimen's S1, measured as
  distance to the three nearest glomeruli (notebook 33's rule).

**3 · Measurement.**
- Gradients are pseudobulk log2 CPM ratios of summed counts per specimen and segment. These are
  not compressed by log1p at low per-structure counts.
- Sensitivities: the per-structure log1p-mean contrast used earlier; tertiles of our human
  expression; genes with equal probe counts in both panels (workstream 3).
- Genes need ≥ 20 summed counts in every specimen × segment pseudobulk of both species, and
  reference mean ≥ 3 log2 units in one of the compared segment groups.

**4 · Tissue state.**
- (a) **Mouse AKI.** IR2A2 and IR2A4 against controls, all labelled by the mouse-only pipeline.
  The amplitude ratio AKI / control is estimated the same way.
- (b) **Injury and stress scores** from canonical PT injury markers and, separately, immediate-early
  genes. Each is the mean within-dataset z of pseudobulk log2 CPM. Across Lake donors
  (healthy, AKI and CKD cortex), we ask whether donor amplitude tracks the injury score. Lake's
  own altered-state fraction (aPT, dPT, cycPT among PT nuclei) is a second score.
- (c) **Within specimen.** Structures below each specimen's median injury score form a
  low-injury half; we compare the human / mouse ratio in those halves.

**5 · Consequence.**
- The flattening model predicts human gradient = r × mouse gradient, with r the primary ratio
  (microdissection male, S2 − S1, OLS).
- A gene contrast is **beyond flattening** when |human − r·mouse| ≥ 0.5 log2 with the same sign
  in all four specimen pairings.
- Beyond-flattening subclasses: reversal, human-specific gradient, human-stronger, other.
- A robust pathway (notebook 37) is beyond flattening when its members' residual |human − r·mouse|
  is enriched against matched genes (joint test, BH ≤ 0.10 within the 36).

**Decision rules (fixed before results):**

| Claim | Supported only if |
|---|---|
| D1 · Global flattening | S2 − S1 OLS ratio < 0.7 with 95% CI excluding 1, against ≥ 2 independent reference groups (microdissection; mouse snRNA; Lake human) |
| D2 · Not log compression | the ratio stays < 0.7 (CI excluding 1) in the top tertile of human expression, among probe-balanced genes, and for the primary reference |
| D3 · Not specific to our platform or donor | Lake healthy human ÷ mouse snRNA (male), referenced on microdissection, is < 0.7 with CI excluding 1 |
| D4 · Region | reported: if S3 − early (cortical mouse S3) ≥ 0.85 while S2 − S1 < 0.7, flattening is S2-specific |
| D5 · Tissue state is a plausible explanation | AKI / control ratio ≤ human / mouse ratio (same reference), **or** Spearman(Lake donor injury score, amplitude) < −0.5, **or** the low-injury-half human / mouse ratio ≥ 0.85 |

## 1 · Folders and settings

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates'
external = data_root / 'external'
output = results_root / 'pt_zonation_amplitude'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'mouse_only_v5' / 'all_mouse_tubules_harmony.h5ad',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            upstream37 / 'tables' / 'table_S2_pathways.csv',
            results_root / 'pt_pathway_remodeling_scfates' / 'pathway_coverage.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz', lake_file, probe_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '39.zonation_amplitude.1'
SEED, N_BOOT = 39, 2000
MIN_COUNTS = 20               # summed counts per specimen x segment pseudobulk, every specimen
REFERENCE_LEVEL = 3.          # reference mean log2 level in one of the compared segment groups
MIN_NUCLEI = 50               # per donor and segment in snRNA references
FLATTENING, CI_LIMIT, REGION_LIMIT, LOW_INJURY_LIMIT, INJURY_RHO = .7, 1., .85, .85, -.5
RESIDUAL, SIGNED, SPECIFIC, ABSENT = .5, .3, .5, .25
INJURY_MARKERS = ['Havcr1', 'Lcn2', 'Vcam1', 'Cd44', 'Krt8', 'Krt18', 'Krt20', 'Sox9', 'Vim', 'Spp1', 'Clu',
                  'Cdkn1a', 'Timp1', 'Lgals3', 'Ccn2', 'Ctgf']
STRESS_MARKERS = ['Fos', 'Fosb', 'Jun', 'Junb', 'Egr1', 'Atf3', 'Ier2', 'Hspa1a', 'Hspa1b', 'Dnajb1']
print('Results folder:', output)

## 2 · Our structures, counts and segment pseudobulks

- **Cross-species data.** All of notebook 13's PT structures with their reviewed S1/S2/S3 labels.
  No coordinate filter is applied.
- **Gene space.** Accepted orthologs measured in all four specimens (mouse symbols).
- **Library size.** The total over that gene space.
- **Mouse-only data** (controls and AKI) come from the mouse-only pipeline's labels and the raw
  tubule-by-gene matrices.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, cached_frame, digest
from pseudospace import zonation_amplitude as amplitude_module
from pseudospace.zonation_amplitude import (group_sums, log_ratio, ols_slope, deming_slope, amplitude_ratio,
                                            bootstrap_ratio, flattening_class)

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()
display(pd.crosstab(specimen, segment))
structure_depth = pd.DataFrame({'specimen': specimen, 'library': library}).groupby('specimen').library.describe()[['count', '50%', 'mean']]
display(structure_depth.rename(columns={'50%': 'median counts per structure'}))

### Cortical-like mouse S3

Depth is the mean distance to the three nearest reviewed glomeruli of the same specimen, with
centroids verified against the fine GeoJSON (notebook 33's rule).

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']
cortical_s3 = np.zeros(len(specimen), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
    cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
segment_region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
print('Cortical-like mouse S3 structures:', {n: int((cortical_s3 & (specimen == n)).sum()) for n in expected['mouse']})

In [ ]:
def specimen_pseudobulk(count_matrix, specimen_labels, segment_labels, genes):
    """Summed counts per specimen and segment (S1, S2, S3, cortical-S3 and early = S1 + S2)."""
    names_, sums = group_sums(count_matrix, np.char.add(np.char.add(specimen_labels.astype(str), '|'), segment_labels.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=genes)
    out = {}
    for name in np.unique(specimen_labels):
        rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
        out[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
        out[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
        out[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
        out[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
        out[(name, 'early')] = out[(name, 'S1')] + out[(name, 'S2')]
    return pd.DataFrame(out)


ours_counts = specimen_pseudobulk(counts, specimen, segment_region, gene_space)
ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(
    segment_region, {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
    for key in ours_counts.columns})

### Mouse controls and AKI from the mouse-only pipeline

The mouse-only object carries its own counts layer and S1/S2/S3 labels for all four mice. Its
structure identifiers predate the current tubule-by-gene matrices, so its own counts are used.
It covers 9,688 genes, of which those in the shared gene space are kept. Controls and AKI are
therefore compared within one labelling and one gene set.

In [ ]:
def mouse_only_counts():
    """PT structures of all four mice from the mouse-only object (its own counts layer and labels)."""
    labelled = ad.read_h5ad(results_root / 'mouse_only_v5' / 'all_mouse_tubules_harmony.h5ad', backed='r')
    obs = labelled.obs[['sample', 'condition', 'segment_class']].copy()
    keep = obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()
    matrix = sparse.csr_matrix(labelled.layers['counts'][np.flatnonzero(keep)], dtype=float)
    var_names = labelled.var_names.copy()
    labelled.file.close()
    index = var_names.get_indexer(gene_space)
    present = index >= 0
    transform = sparse.csr_matrix((np.ones(present.sum()), (index[present], np.flatnonzero(present))),
                                  shape=(len(var_names), len(gene_space)))
    stacked = sparse.csr_matrix(matrix @ transform)
    return {'data': stacked.data, 'indices': stacked.indices, 'indptr': stacked.indptr, 'shape': np.array(stacked.shape),
            'segment': obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()[keep].astype(str),
            'specimen': obs['sample'].astype(str).to_numpy()[keep].astype(str), 'genes_present': np.array([int(present.sum())])}


mouse_payload = cached_payload('mouse_only_counts', mouse_only_counts, root=cache,
                               inputs={'genes': list(gene_space)}, code=code)
mouse_counts = sparse.csr_matrix((mouse_payload['data'], mouse_payload['indices'], mouse_payload['indptr']),
                                 shape=tuple(mouse_payload['shape']))
mouse_specimen, mouse_segment = mouse_payload['specimen'].astype(str), mouse_payload['segment'].astype(str)
mouse_library = np.asarray(mouse_counts.sum(axis=1)).ravel()
mouse_pb = specimen_pseudobulk(mouse_counts, mouse_specimen, mouse_segment, gene_space)
mouse_pb_library = pd.Series({key: float(mouse_library[(mouse_specimen == key[0]) & np.isin(
    mouse_segment, {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
    for key in mouse_pb.columns})
display(pd.crosstab(mouse_specimen, mouse_segment))
print('Shared-space genes present in the mouse-only object:', int(mouse_payload['genes_present'][0]))

## 3 · Our gradients

- **Pseudobulk log2 ratio (primary):** log2 CPM(late) − log2 CPM(early) from summed counts per
  specimen.
- **log1p mean (sensitivity):** mean log1p(CP10k) per segment, the contrast used in notebooks 34
  and 37.
- **Contrasts:**
  - `S2-S1`;
  - `S3c-early`, with mouse S3 restricted to cortical-like structures and human S3 complete (both
    human sections are cortex);
  - `S3-early`, all S3.

In [ ]:
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}


def gradients(pb, pb_library, specimens):
    out = {}
    for contrast, (late, early) in CONTRASTS.items():
        for name in specimens:
            out[(contrast, name)] = log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
    return pd.DataFrame(out, index=pb.index)


ours = gradients(ours_counts, ours_library, names)
mouse_only = gradients(mouse_pb, mouse_pb_library, ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'])
log1p_means = {}
for name in names:
    for label, parts in {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}.items():
        rows = (specimen == name) & np.isin(segment_region, parts)
        log1p_means[(name, label)] = np.asarray(lognorm[rows].mean(axis=0)).ravel()
log1p_means = pd.DataFrame(log1p_means, index=gene_space)
ours_log1p = pd.DataFrame({(contrast, name): log1p_means[(name, late)] - log1p_means[(name, early)]
                           for contrast, (late, early) in CONTRASTS.items() for name in names})


def count_filter(pb, specimens, parts):
    return pb[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)


our_genes = {contrast: gene_space[count_filter(ours_counts, names, list(parts)).to_numpy()]
             for contrast, parts in CONTRASTS.items()}
mouse_only_genes = {contrast: gene_space[count_filter(mouse_pb, ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'], list(parts)).to_numpy()]
                    for contrast, parts in CONTRASTS.items()}
def species_mean(frame, contrast, specimens):
    return frame[[(contrast, n) for n in specimens]].mean(axis=1)


print({c: len(g) for c, g in our_genes.items()}, 'genes pass the count filter (cross-species)')

## 4 · Independent references

Each reference gives a gene-level gradient for each contrast (log2 units), plus a donor-resampling
function for the bootstrap.
- **Microdissection male** (GSE150338): mean log2(TPM + 1) per mouse, then over mice. Segments
  come from different mice, so the bootstrap resamples mice within each segment.
- **Microdissection by sex** (GSE212213): segment means only, so the bootstrap resamples genes only.
- **Mouse snRNA** (Census): log2(CPM + 1) per donor; early = pooled S1 + S2 counts; donors with
  ≥ 50 nuclei in every segment.
- **Lake / KPMP human cortex** (CELLxGENE v1.5): PT-S1/S2/S3 nuclei of cortex samples, summed per
  donor and segment, with genes mapped by Ensembl id. Healthy donors are the human reference. All
  donors, healthy plus AKI and CKD, enter the tissue-state analysis.

For microdissection, "S3c" is the dissected S3, which lies in the outer stripe, so it equals
"S3".

In [ ]:
import re


def microdissection_male():
    table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
    table = table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
    per_mouse = {}
    for column in table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    values = {key: np.log2(table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
              for key, cols in per_mouse.items()}
    return pd.DataFrame(values).reindex(gene_space)


micro_male = microdissection_male()
micro_sex_table = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
micro_sex_table = micro_sex_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_sex = {sex: pd.DataFrame({f'S{k}': np.log2(pd.to_numeric(micro_sex_table[f'{prefix}PTS{k}'], errors='coerce').clip(lower=0) + 1)
                                for k in (1, 2, 3)}).reindex(gene_space)
             for sex, prefix in (('male', ''), ('female', 'f'))}


def census_segments(label, segment_map):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})
census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment, plus PT-state counts per donor."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
        states = obs[cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])]
        state_counts = states.groupby(['donor_id', 'subclass.l2']).size().unstack(fill_value=0)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    state_columns = ['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']
    state_counts = state_counts.reindex(columns=state_columns, fill_value=0)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy(),
            'state_donors': state_counts.index.to_numpy().astype(str), 'state_counts': state_counts.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_states = pd.DataFrame(lake['state_counts'], index=lake['state_donors'], columns=['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])
lake_counts = {}
lake_totals = {}
for donor in lake_donors:
    for seg_name in ('S1', 'S2', 'S3'):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
        lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
        lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)
print(f'Lake cortex donors with ≥ {MIN_NUCLEI} nuclei in each segment: {len(lake_donors)} '
      f'({lake_disease.loc[lake_donors].value_counts().to_dict()}); mouse snRNA donors: {sn_sex.value_counts().to_dict()}')

In [ ]:
def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    """Per-donor log2 CPM contrast and per-segment levels; early pools S1 + S2 counts."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def reference_from_donors(count_frame, totals, donors, contrast):
    """Reference gradient (mean over donors), expressed-gene mask and a donor-resampling draw function."""
    return donor_reference(*donor_contrasts(count_frame, totals, donors, contrast))


def reference_from_census_human():
    """Census human cortex: S3 minus convoluted PT per donor (only for the S3 contrasts)."""
    donors = census_h_sex.index
    late = {d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}
    early = {d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}
    per_donor = pd.DataFrame({d: late[d] - early[d] for d in donors}).T
    level = pd.DataFrame({d: np.maximum(late[d], early[d]) for d in donors}).T
    return donor_reference(per_donor, level)


def donor_reference(per_donor, level):
    expressed = level.mean().ge(REFERENCE_LEVEL)

    def draw(rng):
        rows = np.arange(len(per_donor)) if rng is None else rng.integers(0, len(per_donor), len(per_donor))
        return per_donor.iloc[rows].mean()[expressed]

    error_variance = float((per_donor.loc[:, expressed].var(ddof=1) / len(per_donor)).mean())
    return {'gradient': per_donor.mean()[expressed], 'draw': draw, 'error_variance': error_variance,
            'n_donors': len(per_donor), 'per_donor': per_donor}


def reference_from_micro_male(contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    parts_early = ['S1', 'S2'] if early == 'early' else [early]
    by_segment = {s: micro_male[[c for c in micro_male.columns if c[0] == s]] for s in ('S1', 'S2', 'S3')}

    def gradient(rng):
        means = {}
        for s, frame in by_segment.items():
            pick = np.arange(frame.shape[1]) if rng is None else rng.integers(0, frame.shape[1], frame.shape[1])
            means[s] = frame.iloc[:, pick].mean(axis=1)
        return means[late] - sum(means[p] for p in parts_early) / len(parts_early), means

    observed, means = gradient(None)
    expressed = pd.concat([means[late], *[means[p] for p in parts_early]], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    error_variance = float(sum((by_segment[s].var(axis=1, ddof=1) / by_segment[s].shape[1] * (1 if s == late else 1 / len(parts_early) ** 2))
                               for s in {late, *parts_early})[expressed].mean())
    return {'gradient': observed[expressed], 'draw': lambda rng: gradient(rng)[0][expressed],
            'error_variance': error_variance, 'n_donors': int(min(f.shape[1] for f in by_segment.values()))}


def reference_from_micro_sex(sex, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    table = micro_sex[sex]
    early_value = table[['S1', 'S2']].mean(axis=1) if early == 'early' else table[early]
    gradient = table[late] - early_value
    expressed = pd.concat([table[late], early_value], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    return {'gradient': gradient[expressed], 'draw': lambda rng: gradient[expressed], 'error_variance': np.nan, 'n_donors': 0}


def build_references(contrast):
    refs = {'microdissection male (GSE150338)': ('microdissection', reference_from_micro_male(contrast)),
            'microdissection male (GSE212213)': ('microdissection', reference_from_micro_sex('male', contrast)),
            'microdissection female (GSE212213)': ('microdissection', reference_from_micro_sex('female', contrast)),
            'mouse snRNA male (Census)': ('mouse snRNA', reference_from_donors(sn_counts, sn_totals, sn_sex.index[sn_sex.eq('male')], contrast)),
            'mouse snRNA female (Census)': ('mouse snRNA', reference_from_donors(sn_counts, sn_totals, sn_sex.index[sn_sex.eq('female')], contrast)),
            'Lake healthy human cortex': ('human snRNA', reference_from_donors(lake_counts, lake_totals,
                                                                             [d for d in lake_donors if lake_disease[d] == 'normal'], contrast))}
    if contrast != 'S2-S1':
        refs['Census human cortex'] = ('human snRNA', reference_from_census_human())
    return refs


references = {contrast: build_references(contrast) for contrast in ('S2-S1', 'S3c-early', 'S3-early')}
display(pd.DataFrame({(c, k): {'group': g, 'genes expressed': int(len(r['gradient'])), 'donors or mice': r['n_donors']}
                      for c, refs in references.items() for k, (g, r) in refs.items()}).T)

## 5 · Dilution-free amplitude ratios

For each contrast and reference, using genes that pass our count filter and are expressed in the
reference:
- the slope of our human gradient (mean of the two sections) and of our mouse gradient (mean of
  the two mice) on the reference, by OLS (primary), Deming and orthogonal regression;
- the ratio of the two slopes;
- a two-stage bootstrap (donors, then genes) and a genes-only bootstrap;
- the four specimen-pairing ratios.

**Deming error-variance ratio.**
- Our error variance is the between-specimen variance of the gradient ÷ 2, averaged over genes.
- The reference error variance is the between-donor variance ÷ donors, averaged over genes.

In [ ]:
def our_error_variance(frame, contrast, specimens, genes):
    values = frame.loc[genes, [(contrast, n) for n in specimens]]
    return float((values.var(axis=1, ddof=1) / len(specimens)).mean())


def ratio_table(frame, contrast, refs, genes_ok, human_ids, mouse_ids, *, label, n_boot=N_BOOT, methods=True):
    rows = []
    human = species_mean(frame, contrast, human_ids)
    mouse = species_mean(frame, contrast, mouse_ids)
    for name, (group, ref) in refs.items():
        genes = ref['gradient'].index.intersection(genes_ok)
        ref_series = ref['gradient'].loc[genes]
        ols = amplitude_ratio(human.loc[genes], mouse.loc[genes], ref_series)
        (lo2, _, hi2), _ = bootstrap_ratio(human.loc[genes], mouse.loc[genes], lambda rng: ref['draw'](rng).reindex(genes),
                                           n_boot=n_boot, seed=SEED)
        (lo1, _, hi1), _ = bootstrap_ratio(human.loc[genes], mouse.loc[genes], lambda rng: ref_series, n_boot=n_boot, seed=SEED)
        pair_ratios = [ols_slope(ref_series, frame.loc[genes, (contrast, h)]) / ols_slope(ref_series, frame.loc[genes, (contrast, m)])
                       for h in human_ids for m in mouse_ids]
        row = {'analysis': label, 'contrast': contrast, 'reference': name, 'group': group, 'genes': len(genes),
               'slope_human': ols['slope_human'], 'slope_mouse': ols['slope_mouse'], 'ratio_ols': ols['ratio'],
               'ci_low': lo2, 'ci_high': hi2, 'ci_low_genes_only': lo1, 'ci_high_genes_only': hi1,
               'pairing_min': min(pair_ratios), 'pairing_max': max(pair_ratios)}
        if methods:
            row['ratio_orthogonal'] = amplitude_ratio(human.loc[genes], mouse.loc[genes], ref_series, method='orthogonal')['ratio']
            if np.isfinite(ref['error_variance']):
                deltas = (our_error_variance(frame, contrast, human_ids, genes) / ref['error_variance'],
                          our_error_variance(frame, contrast, mouse_ids, genes) / ref['error_variance'])
                row['ratio_deming'] = amplitude_ratio(human.loc[genes], mouse.loc[genes], ref_series, method='deming',
                                                      delta_human=deltas[0], delta_mouse=deltas[1])['ratio']
                (dlo, _, dhi), _ = bootstrap_ratio(human.loc[genes], mouse.loc[genes], lambda rng: ref['draw'](rng).reindex(genes),
                                                   n_boot=500, seed=SEED, method='deming', deltas=deltas)
                row['deming_ci_low'], row['deming_ci_high'] = dlo, dhi
        rows.append(row)
    return pd.DataFrame(rows)


main_ratios = cached_frame('main_ratios', lambda: pd.concat(
    [ratio_table(ours, contrast, references[contrast], our_genes[contrast], expected['human'], expected['mouse'], label='human / mouse')
     for contrast in ('S2-S1', 'S3c-early', 'S3-early')], ignore_index=True),
    root=cache, params={'n_boot': N_BOOT, 'seed': SEED}, inputs={'ours': ours, 'genes': {k: list(v) for k, v in our_genes.items()}}, code=code)
naive = {contrast: ols_slope(species_mean(ours, contrast, expected['mouse']).loc[our_genes[contrast]],
                             species_mean(ours, contrast, expected['human']).loc[our_genes[contrast]]) for contrast in CONTRASTS}
main_ratios.to_csv(tables / 'amplitude_ratios.csv', index=False)
display(main_ratios[['contrast', 'reference', 'genes', 'slope_human', 'slope_mouse', 'ratio_ols', 'ci_low', 'ci_high',
                     'pairing_min', 'pairing_max', 'ratio_deming', 'ratio_orthogonal']].round(3))
print('Naive slope of our human on our mouse gradient (dilution-prone):', {k: round(v, 3) for k, v in naive.items()})

### External check: is flattening present in independent human data?

Lake healthy human and Census mouse snRNA gradients are both regressed on male mouse
microdissection. The ratio is a human / mouse amplitude ratio that uses no data from our cohort.
As a sensitivity, the same is done with microdissection as the "mouse" and mouse snRNA as the
reference.

In [ ]:
def external_ratio(contrast, human_ref, mouse_ref, axis_ref, n_boot=N_BOOT):
    genes = human_ref['gradient'].index.intersection(mouse_ref['gradient'].index).intersection(axis_ref['gradient'].index)
    point = amplitude_ratio(human_ref['gradient'].loc[genes], mouse_ref['gradient'].loc[genes], axis_ref['gradient'].loc[genes])
    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(n_boot):
        pick = genes[rng.integers(0, len(genes), len(genes))]
        h, m, a = (ref['draw'](rng).reindex(genes).loc[pick].to_numpy() for ref in (human_ref, mouse_ref, axis_ref))
        draws.append(amplitude_ratio(h, m, a)['ratio'])
    low, high = np.nanpercentile(draws, [2.5, 97.5])
    return {'contrast': contrast, 'genes': len(genes), 'ratio_ols': point['ratio'], 'ci_low': low, 'ci_high': high}


external_rows = []
for contrast in ('S2-S1', 'S3c-early'):
    refs = references[contrast]
    external_rows.append({'analysis': 'Lake healthy ÷ mouse snRNA male, on microdissection male',
                          **external_ratio(contrast, refs['Lake healthy human cortex'][1], refs['mouse snRNA male (Census)'][1],
                                           refs['microdissection male (GSE150338)'][1])})
    external_rows.append({'analysis': 'Lake healthy ÷ microdissection male, on mouse snRNA male',
                          **external_ratio(contrast, refs['Lake healthy human cortex'][1], refs['microdissection male (GSE150338)'][1],
                                           refs['mouse snRNA male (Census)'][1])})
external_ratios = pd.DataFrame(external_rows)
external_ratios.to_csv(tables / 'external_amplitude_ratios.csv', index=False)
display(external_ratios.round(3))

## 6 · Measurement checks

Each check uses the S2 − S1 contrast and the primary references:
- ratio within tertiles of our human expression (mean pseudobulk log2 CPM over S1 and S2);
- the per-structure log1p-mean contrast;
- genes with equal probe counts in both Visium HD panels.

In [ ]:
probe = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = probe.index[probe.probe_balance.eq('balanced')]
human_level = pd.concat([log2_cpm(ours_counts[(n, p)], ours_library[(n, p)]) for n in expected['human'] for p in ('S1', 'S2')],
                        axis=1).mean(axis=1)
primary_refs = {k: references['S2-S1'][k] for k in ('microdissection male (GSE150338)', 'mouse snRNA male (Census)', 'Lake healthy human cortex')}
check_parts = []
genes_s2 = our_genes['S2-S1']
tertile = pd.qcut(human_level.loc[genes_s2], 3, labels=['low', 'middle', 'high'])
for label in ('low', 'middle', 'high'):
    check_parts.append(ratio_table(ours, 'S2-S1', primary_refs, tertile.index[tertile.eq(label)], expected['human'], expected['mouse'],
                                   label=f'human expression tertile: {label}', n_boot=1000, methods=False))
check_parts.append(ratio_table(ours_log1p, 'S2-S1', primary_refs, genes_s2, expected['human'], expected['mouse'],
                               label='per-structure log1p mean', n_boot=1000, methods=False))
check_parts.append(ratio_table(ours, 'S2-S1', primary_refs, genes_s2.intersection(balanced), expected['human'], expected['mouse'],
                               label='equal probe counts', n_boot=1000, methods=False))
measurement_checks = pd.concat(check_parts, ignore_index=True)
measurement_checks.to_csv(tables / 'measurement_checks.csv', index=False)
display(measurement_checks[['analysis', 'reference', 'genes', 'ratio_ols', 'ci_low', 'ci_high']].round(3))

## 7 · Tissue state

### (a) Does acute kidney injury flatten mouse zonation?

All four mice are labelled by the same mouse-only pipeline. The AKI / control ratio uses the same
references and bootstrap. Because these labels differ from the cross-species ones, the control
amplitude is recomputed here: the human / control ratio uses mouse-only labels for the mice.

In [ ]:
aki_ratios = pd.concat([ratio_table(mouse_only, contrast, references[contrast], mouse_only_genes[contrast],
                                    ['IR2A2', 'IR2A4'], ['Ctrl1A2', 'Ctrl1A4'], label='AKI / control', methods=False)
                        for contrast in ('S2-S1', 'S3-early')], ignore_index=True)
aki_ratios.to_csv(tables / 'aki_amplitude_ratios.csv', index=False)
display(aki_ratios[['contrast', 'reference', 'genes', 'ratio_ols', 'ci_low', 'ci_high', 'pairing_min', 'pairing_max']].round(3))

### (b) Injury and stress scores, and donor-level amplitude in Lake

Each score is the mean, over markers present in the gene space, of the within-dataset z of
pseudobulk log2 CPM (all PT segments pooled). Scores compare specimens or donors **within** a
dataset; they are not comparable across platforms. The Lake altered-state fraction is
(aPT + dPT + cycPT) ÷ all PT nuclei in cortex.

Each Lake donor's amplitude is the OLS slope of that donor's S2 − S1 gradient on the male mouse
microdissection reference, using genes expressed in both.

In [ ]:
present_injury = [g for g in INJURY_MARKERS if g in set(gene_space)]
present_stress = [g for g in STRESS_MARKERS if g in set(gene_space)]
print('Injury markers present:', present_injury)
print('Stress markers present:', present_stress)


def marker_score(levels, markers):
    block = levels[markers]
    z = (block - block.mean()) / block.std(ddof=1).replace(0, np.nan)
    return z.mean(axis=1)


def specimen_levels(pb, pb_library, specimens):
    return pd.DataFrame({n: log2_cpm(pb[(n, 'S1')] + pb[(n, 'S2')] + pb[(n, 'S3')],
                                     pb_library[(n, 'S1')] + pb_library[(n, 'S2')] + pb_library[(n, 'S3')]) for n in specimens}).T


mouse_levels = specimen_levels(mouse_pb, mouse_pb_library, ['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'])
human_levels = specimen_levels(ours_counts, ours_library, expected['human'])
lake_levels = pd.DataFrame({d: log2_cpm(sum(lake_counts[(d, s)] for s in ('S1', 'S2', 'S3')),
                                        sum(lake_totals[(d, s)] for s in ('S1', 'S2', 'S3'))) for d in lake_donors}).T
mouse_scores = pd.DataFrame({'injury': marker_score(mouse_levels, present_injury), 'stress': marker_score(mouse_levels, present_stress)})
marker_shift = (mouse_levels.loc[['IR2A2', 'IR2A4'], present_injury + present_stress].mean()
                - mouse_levels.loc[['Ctrl1A2', 'Ctrl1A4'], present_injury + present_stress].mean())
micro_ref = references['S2-S1']['microdissection male (GSE150338)'][1]
lake_per_donor, _ = donor_contrasts(lake_counts, lake_totals, lake_donors, 'S2-S1')
lake_amp_genes = micro_ref['gradient'].index.intersection(lake_per_donor.columns[lake_per_donor.notna().all()])
lake_scores = pd.DataFrame({
    'disease': lake_disease.loc[lake_donors],
    'injury': marker_score(lake_levels, present_injury), 'stress': marker_score(lake_levels, present_stress),
    'altered_fraction': (lake_states[['aPT', 'dPT', 'cycPT']].sum(axis=1) / lake_states.sum(axis=1)).reindex(lake_donors),
    'amplitude_S2_S1': [ols_slope(micro_ref['gradient'].loc[lake_amp_genes], lake_per_donor.loc[d, lake_amp_genes]) for d in lake_donors]})
from scipy.stats import spearmanr, mannwhitneyu
lake_tests = pd.Series({
    'Spearman injury vs amplitude (all donors)': spearmanr(lake_scores.injury, lake_scores.amplitude_S2_S1).statistic,
    'Spearman altered fraction vs amplitude (all donors)': spearmanr(lake_scores.altered_fraction, lake_scores.amplitude_S2_S1).statistic,
    'Spearman stress vs amplitude (all donors)': spearmanr(lake_scores.stress, lake_scores.amplitude_S2_S1).statistic,
    'Spearman injury vs amplitude (healthy donors)': spearmanr(lake_scores.query('disease == "normal"').injury,
                                                               lake_scores.query('disease == "normal"').amplitude_S2_S1).statistic,
    'median amplitude, healthy': lake_scores.query('disease == "normal"').amplitude_S2_S1.median(),
    'median amplitude, AKI or CKD': lake_scores.query('disease != "normal"').amplitude_S2_S1.median(),
    'Mann–Whitney p, healthy vs disease amplitude': mannwhitneyu(lake_scores.query('disease == "normal"').amplitude_S2_S1,
                                                                 lake_scores.query('disease != "normal"').amplitude_S2_S1).pvalue,
    'n donors': len(lake_scores)})
lake_scores.to_csv(tables / 'lake_donor_scores.csv')
mouse_scores.to_csv(tables / 'mouse_specimen_scores.csv')
display(mouse_scores.round(2))
display(marker_shift.round(2).rename('AKI − control, log2 CPM').to_frame().T)
display(lake_scores.round(3))
display(lake_tests.to_frame('value').round(3))

### (c) Low-injury structures within each specimen

A structure's injury score is log1p of its summed injury-marker counts per 10⁴. Within each
specimen, structures at or below the specimen's median score form the low-injury half. The human
/ mouse S2 − S1 ratio is re-estimated within the low halves and within the high halves.

In [ ]:
marker_index = gene_space.get_indexer(present_injury)
structure_injury = np.log1p(np.asarray(counts[:, marker_index].sum(axis=1)).ravel() / library * 1e4)
half = np.empty(len(specimen), dtype=object)
for name in names:
    mask = specimen == name
    half[mask] = np.where(structure_injury[mask] <= np.median(structure_injury[mask]), 'low', 'high')
half_rows = []
for label in ('low', 'high'):
    keep = half == label
    pb = specimen_pseudobulk(counts[keep], specimen[keep], segment_region[keep], gene_space)
    lib = pd.Series({key: float(library[keep][(specimen[keep] == key[0]) & np.isin(
        segment_region[keep], {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
        for key in pb.columns})
    frame = gradients(pb, lib, names)
    genes = gene_space[count_filter(pb, names, ['S2', 'S1']).to_numpy()]
    half_rows.append(ratio_table(frame, 'S2-S1', primary_refs, genes, expected['human'], expected['mouse'],
                                 label=f'{label}-injury half', n_boot=1000, methods=False))
injury_halves = pd.concat(half_rows, ignore_index=True)
injury_halves.to_csv(tables / 'injury_halves.csv', index=False)
display(injury_halves[['analysis', 'reference', 'genes', 'ratio_ols', 'ci_low', 'ci_high']].round(3))

## 8 · Decisions

In [ ]:
s21 = main_ratios[main_ratios.contrast.eq('S2-S1')].set_index('reference')
passing = s21[(s21.ratio_ols < FLATTENING) & (s21.ci_high < CI_LIMIT)]
D1 = passing.group.nunique() >= 2
primary_name = 'microdissection male (GSE150338)'
checks_primary = measurement_checks[measurement_checks.reference.eq(primary_name)].set_index('analysis')
D2 = bool((checks_primary.loc[['human expression tertile: high', 'equal probe counts'], 'ratio_ols'] < FLATTENING).all()
          and (checks_primary.loc[['human expression tertile: high', 'equal probe counts'], 'ci_high'] < CI_LIMIT).all())
ext = external_ratios.set_index(['analysis', 'contrast']).loc[('Lake healthy ÷ mouse snRNA male, on microdissection male', 'S2-S1')]
D3 = bool(ext.ratio_ols < FLATTENING and ext.ci_high < CI_LIMIT)
s3c = main_ratios[main_ratios.contrast.eq('S3c-early')].set_index('reference')
D4_s2_specific = bool(s3c.loc[primary_name, 'ratio_ols'] >= REGION_LIMIT and s21.loc[primary_name, 'ratio_ols'] < FLATTENING)
human_ratio = s21.loc[primary_name, 'ratio_ols']
aki = aki_ratios[aki_ratios.contrast.eq('S2-S1')].set_index('reference')
low_half = injury_halves[injury_halves.analysis.eq('low-injury half')].set_index('reference')
D5_parts = {'AKI lowers mouse amplitude to or below the human ratio': bool(aki.loc[primary_name, 'ratio_ols'] <= human_ratio),
            'Lake donor injury score tracks amplitude (rho < -0.5)': bool(lake_tests['Spearman injury vs amplitude (all donors)'] < INJURY_RHO),
            'flattening disappears in low-injury structures (ratio ≥ 0.85)': bool(low_half.loc[primary_name, 'ratio_ols'] >= LOW_INJURY_LIMIT)}
D5 = any(D5_parts.values())
decisions = pd.Series({
    'D1 global flattening (S2 − S1, ≥ 2 reference groups)': D1,
    'reference groups passing': ', '.join(sorted(passing.group.unique())),
    'D2 not log compression': D2, 'D3 present in independent human data': D3,
    'D4 flattening S2-specific (region)': D4_s2_specific,
    **{f'D5 · {k}': v for k, v in D5_parts.items()}, 'D5 tissue state plausible': D5,
    'primary ratio r (microdissection male, S2 − S1)': round(float(human_ratio), 3)})
decisions.to_csv(tables / 'decisions.csv')
display(decisions.to_frame('value'))

## 9 · Consequence: which robust pathways and lead genes go beyond global flattening?

Gene contrasts are classified with r = the primary ratio, on S2 − S1 and on S3c − early. A gene
is beyond flattening if either contrast is beyond. Pathway test: the gene statistic is the
largest |human − r·mouse| over the two contrasts. It is compared with matched random genes
(strata: mean expression, detection, coverage), using notebook 37's variance inflation (joint
test), with BH within the 36 robust pathways.

In [ ]:
import ast
from pseudospace.pathway_remodeling import gene_covariates, matching_strata, matched_pathway_tests
from pseudospace.pathway_decoys import joint_matched_test
from statsmodels.stats.multitest import multipletests

r = float(human_ratio)
classified = {}
for contrast in ('S2-S1', 'S3c-early'):
    genes = our_genes[contrast]
    rows = {}
    for g in genes:
        klass, residual = flattening_class(ours.loc[g, [(contrast, n) for n in expected['human']]].to_numpy(),
                                           ours.loc[g, [(contrast, n) for n in expected['mouse']]].to_numpy(), r,
                                           residual=RESIDUAL, signed=SIGNED, specific=SPECIFIC, absent=ABSENT)
        rows[g] = (klass, residual)
    classified[contrast] = pd.DataFrame(rows, index=['class', 'residual']).T
gene_classes = pd.concat({c: f for c, f in classified.items()}, axis=1)
gene_classes.columns = [f'{a}_{b}' for a, b in gene_classes.columns]
gene_classes['beyond_flattening'] = gene_classes[['S2-S1_class', 'S3c-early_class']].apply(
    lambda row: any(isinstance(v, str) and v not in ('consistent with flattening', 'not testable') for v in row), axis=1)
gene_classes['max_abs_residual'] = gene_classes[['S2-S1_residual', 'S3c-early_residual']].astype(float).abs().max(axis=1)
gene_classes.to_csv(tables / 'gene_flattening_classes.csv')
display(pd.DataFrame({c: gene_classes[f'{c}_class'].value_counts() for c in ('S2-S1', 'S3c-early')}).fillna(0).astype(int))

final37 = pd.read_csv(upstream37 / 'tables' / 'table_S2_pathways.csv', index_col=0)
robust_ids = final37.index[final37.robust.astype(bool)].tolist()
coverage = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'pathway_coverage.csv').set_index('pathway_id')
statistic = gene_classes['max_abs_residual'].dropna()
covariate_frame = gene_covariates(lognorm[:, gene_space.get_indexer(statistic.index)], adata.obs.shared_pseudospace.to_numpy(float),
                                  adata.obs.comparison_species.astype(str).eq('human').to_numpy(), specimen, statistic.index).set_index('gene')
residual_strata = matching_strata(covariate_frame, bins=3)
sets = {p: [g for g in ast.literal_eval(coverage.loc[p, 'genes_present']) if g in set(statistic.index)] for p in robust_ids}
sets = {p: v for p, v in sets.items() if len(v) >= 5}
residual_tests = joint_matched_test(cached_frame('residual_matched', lambda: matched_pathway_tests(
    pd.DataFrame({'abs_residual': statistic}), sets, residual_strata, n_null=9999, seed=SEED), root=cache,
    inputs={'statistic': statistic, 'sets': sets}, code=code), final37.vif, keys=()).set_index('pathway_id')
residual_tests['q_within_robust'] = multipletests(residual_tests.p_joint, method='fdr_bh')[1]
pathway_classes = pd.DataFrame({
    'name': final37.loc[list(sets), 'name'], 'program': final37.loc[list(sets), 'program'], 'core': final37.loc[list(sets), 'core'],
    'human_high_at_peak': final37.loc[list(sets), 'human_high_at_peak'],
    'residual_effect': residual_tests.effect, 'q': residual_tests.q_within_robust,
    'members_tested': [len(v) for v in sets.values()],
    'members_beyond_flattening': [int(gene_classes.loc[v, 'beyond_flattening'].fillna(False).astype(bool).sum()) for v in sets.values()]})
pathway_classes['class'] = np.where(pathway_classes.q.le(.10) & pathway_classes.residual_effect.gt(0),
                                    'beyond flattening', 'consistent with flattening')
pathway_classes.sort_values(['class', 'q']).to_csv(tables / 'robust_pathway_flattening_classes.csv')
display(pathway_classes.sort_values(['class', 'q']).round(3))

lead_genes = ['Gatm', 'Acadm', 'Hadh', 'Acaa2', 'Acads', 'Acadl', 'Gclc', 'Gclm', 'Gss', 'Hmgcs1', 'Cyp51', 'Hmgcr', 'Sqle',
              'Slc22a6', 'Slc22a8', 'Slc13a3', 'Slc6a19', 'Dcxr', 'Pah', 'Igfbp4', 'Glyat', 'Ugt3a1', 'Acox2', 'Nt5e', 'Cyp24a1',
              'Phgdh', 'Psat1', 'Ugt1a9', 'Ephx1', 'Rbp4', 'Aox1', 'Inmt', 'Me1', 'Lpl', 'Pank1', 'Coasy', 'Cyp2e1', 'Acsm3',
              'Crot', 'Nudt19', 'Slc27a2', 'Cyp7b1', 'Cyp4b1', 'Slc22a7', 'Slc22a13', 'Slc5a8', 'Hsd11b1', 'Adh1']
lead_table = pd.DataFrame({g: {
    'human S2−S1': species_mean(ours, 'S2-S1', expected['human']).get(g, np.nan),
    'mouse S2−S1': species_mean(ours, 'S2-S1', expected['mouse']).get(g, np.nan),
    'human S3c−early': species_mean(ours, 'S3c-early', expected['human']).get(g, np.nan),
    'mouse S3c−early': species_mean(ours, 'S3c-early', expected['mouse']).get(g, np.nan),
    'S2−S1 class': gene_classes['S2-S1_class'].get(g, 'not testable'),
    'S3c−early class': gene_classes['S3c-early_class'].get(g, 'not testable')} for g in lead_genes if g in set(gene_space)}).T
lead_table.to_csv(tables / 'lead_gene_flattening_classes.csv')
display(lead_table.round(2))

## 9b · Post-hoc diagnostics (not pre-registered): flatter, or differently zonated?

The pre-registered ratios split by reference species. Against every **mouse** reference, our
human / mouse ratio is about 0.15 (S2 − S1). Against the **human** references it is about 2–3.
The protocol anticipated a bias in each direction, but not one this large. When the two
directions disagree this much, the dominant pattern is not a global loss of amplitude. It is that
the two species zonate **different genes**.

Four diagnostics follow, written after the decisions above and not used to change them:
- **P1, symmetric external ratio.** Human and mouse snRNA amplitudes compared on a **human**
  axis (Census human cortex; S3 contrasts only, the only ones it resolves), to mirror the
  pre-registered check on a mouse axis.
- **P2, reference-free amplitude.** The noise-corrected SD of the S2 − S1 gradient across genes,
  in each species. Signal variance is the variance of the mean gradient minus the mean per-gene
  sampling variance; replicates are specimens for us and donors externally. This asks whether
  human S1 and S2 differ less **overall**, whichever genes carry the difference.
- **P3, disattenuated cross-species correlation** of S2 − S1 gradients: correlation ÷
  √(reliability_human × reliability_mouse). A low value means the zonation programs are largely
  species-specific.
- **P4, Lake donor amplitude on a human axis.** Each donor's S2 − S1 gradient regressed on the
  mean of the **other** healthy Lake donors, against the donor injury score. This is a
  sensitivity for D5(b), whose pre-registered amplitude used a mouse axis.

In [ ]:
def signal_sd(per_replicate):
    """Noise-corrected SD across genes of the replicate-mean gradient (replicates x genes)."""
    values = per_replicate.to_numpy(float)
    mean = values.mean(axis=0)
    noise = values.var(axis=0, ddof=1).mean() / values.shape[0]
    return float(np.sqrt(max(mean.var(ddof=1) - noise, 0))), float(mean.var(ddof=1)), float(noise)


def amplitude_and_correlation(human_reps, mouse_reps, n_boot=1000):
    genes = human_reps.columns.intersection(mouse_reps.columns)
    h, m = human_reps[genes], mouse_reps[genes]

    def stats(cols):
        sh, vh, nh = signal_sd(h[cols])
        sm, vm, nm = signal_sd(m[cols])
        rel_h, rel_m = max(1 - nh / vh, 1e-9), max(1 - nm / vm, 1e-9)
        corr = np.corrcoef(h[cols].mean(), m[cols].mean())[0, 1]
        return sh / sm if sm > 0 else np.nan, corr / np.sqrt(rel_h * rel_m), corr, rel_h, rel_m

    point = stats(genes)
    rng = np.random.default_rng(SEED)
    draws = np.array([stats(genes[rng.integers(0, len(genes), len(genes))])[:2] for _ in range(n_boot)])
    return {'genes': len(genes), 'signal_sd_ratio': point[0], 'sd_ratio_ci_low': np.percentile(draws[:, 0], 2.5),
            'sd_ratio_ci_high': np.percentile(draws[:, 0], 97.5), 'disattenuated_r': point[1],
            'r_ci_low': np.percentile(draws[:, 1], 2.5), 'r_ci_high': np.percentile(draws[:, 1], 97.5),
            'raw_r': point[2], 'reliability_human': point[3], 'reliability_mouse': point[4]}


ours_reps_h = ours[[('S2-S1', n) for n in expected['human']]].T.reset_index(drop=True).loc[:, our_genes['S2-S1']]
ours_reps_m = ours[[('S2-S1', n) for n in expected['mouse']]].T.reset_index(drop=True).loc[:, our_genes['S2-S1']]
lake_reps = references['S2-S1']['Lake healthy human cortex'][1]['per_donor']
sn_male_reps = references['S2-S1']['mouse snRNA male (Census)'][1]['per_donor']
shared_external = references['S2-S1']['Lake healthy human cortex'][1]['gradient'].index.intersection(
    references['S2-S1']['mouse snRNA male (Census)'][1]['gradient'].index)
def replicate_gradients(frame, contrast, specimens, genes):
    return frame[[(contrast, n) for n in specimens]].T.reset_index(drop=True).loc[:, genes]


posthoc_amplitude = pd.DataFrame({
    'S2−S1 · ours: human sections vs control mice': amplitude_and_correlation(ours_reps_h, ours_reps_m),
    'S2−S1 · external: Lake healthy vs male mouse snRNA': amplitude_and_correlation(lake_reps[shared_external],
                                                                                   sn_male_reps[shared_external]),
    'S2−S1 · mouse AKI vs control (mouse-only labels)': amplitude_and_correlation(
        replicate_gradients(mouse_only, 'S2-S1', ['IR2A2', 'IR2A4'], mouse_only_genes['S2-S1']),
        replicate_gradients(mouse_only, 'S2-S1', ['Ctrl1A2', 'Ctrl1A4'], mouse_only_genes['S2-S1'])),
    'S3c−early · ours: human sections vs control mice (cortical S3)': amplitude_and_correlation(
        replicate_gradients(ours, 'S3c-early', expected['human'], our_genes['S3c-early']),
        replicate_gradients(ours, 'S3c-early', expected['mouse'], our_genes['S3c-early'])),
    'S3−early · mouse AKI vs control (mouse-only labels)': amplitude_and_correlation(
        replicate_gradients(mouse_only, 'S3-early', ['IR2A2', 'IR2A4'], mouse_only_genes['S3-early']),
        replicate_gradients(mouse_only, 'S3-early', ['Ctrl1A2', 'Ctrl1A4'], mouse_only_genes['S3-early']))}).T
posthoc_amplitude.to_csv(tables / 'posthoc_reference_free_amplitude.csv')
display(posthoc_amplitude.round(3))

symmetric_rows = []
for contrast in ('S3-early',):  # references do not distinguish cortical S3, so one S3 contrast suffices
    refs = references[contrast]
    symmetric_rows.append({'analysis': 'Lake healthy ÷ mouse snRNA male, on Census human axis',
                           **external_ratio(contrast, refs['Lake healthy human cortex'][1], refs['mouse snRNA male (Census)'][1],
                                            refs['Census human cortex'][1], n_boot=1000)})
    symmetric_rows.append({'analysis': 'Lake healthy ÷ mouse snRNA male, on microdissection male axis',
                           **external_ratio(contrast, refs['Lake healthy human cortex'][1], refs['mouse snRNA male (Census)'][1],
                                            refs['microdissection male (GSE150338)'][1], n_boot=1000)})
symmetric_external = pd.DataFrame(symmetric_rows)
symmetric_external.to_csv(tables / 'posthoc_symmetric_external_ratios.csv', index=False)
display(symmetric_external.round(3))

healthy = [d for d in lake_donors if lake_disease[d] == 'normal']
loo_amplitude = {}
for donor in lake_donors:
    others = [d for d in healthy if d != donor]
    axis = lake_per_donor.loc[others].mean()
    genes = references['S2-S1']['Lake healthy human cortex'][1]['gradient'].index
    loo_amplitude[donor] = ols_slope(axis.loc[genes], lake_per_donor.loc[donor, genes])
lake_scores['amplitude_S2_S1_human_axis'] = pd.Series(loo_amplitude)
posthoc_lake = pd.Series({
    'Spearman injury vs human-axis amplitude (all donors)': spearmanr(lake_scores.injury, lake_scores.amplitude_S2_S1_human_axis).statistic,
    'Spearman altered fraction vs human-axis amplitude': spearmanr(lake_scores.altered_fraction, lake_scores.amplitude_S2_S1_human_axis).statistic,
    'median human-axis amplitude, healthy': lake_scores.query('disease == "normal"').amplitude_S2_S1_human_axis.median(),
    'median human-axis amplitude, AKI or CKD': lake_scores.query('disease != "normal"').amplitude_S2_S1_human_axis.median()})
lake_scores.to_csv(tables / 'lake_donor_scores.csv')
display(posthoc_lake.to_frame('value').round(3))

expression_status = pd.Series({g: ('not expressed in mouse' if ours_counts.loc[g, [(n, p) for n in expected['mouse'] for p in ('S1', 'S2')]].min() < MIN_COUNTS
                                   and ours_counts.loc[g, [(n, p) for n in expected['human'] for p in ('S1', 'S2')]].min() >= MIN_COUNTS
                                   else 'not expressed in human' if ours_counts.loc[g, [(n, p) for n in expected['human'] for p in ('S1', 'S2')]].min() < MIN_COUNTS
                                   and ours_counts.loc[g, [(n, p) for n in expected['mouse'] for p in ('S1', 'S2')]].min() >= MIN_COUNTS
                                   else 'expressed in both or neither') for g in lead_table.index})
lead_table['expression (S1, S2 counts ≥ 20)'] = expression_status
lead_table.to_csv(tables / 'lead_gene_flattening_classes.csv')

### Reading the results

**What the pre-registered rules say.**
- D1 is met as written: two mouse reference groups give ratios near 0.15.
- The human references (Lake, Census human) give ratios of about 2–3.
- The same split appears with no data from our cohort. Lake ÷ mouse snRNA is about 0.14 on a
  mouse axis and about 1.7 on a human axis.
- D1 as written cannot tell a global loss of amplitude from species-specific zonation. Only the
  bracketing can, and here it is very wide.

**What the post-hoc diagnostics add.**
- **The two species zonate largely different genes.** The disattenuated cross-species correlation
  of S2 − S1 gradients is low, both in our data and in independent snRNA.
- **There is also a real but moderate global reduction.** The noise-corrected SD of human S2 − S1
  gradients is roughly half the mouse value, again in our data and in independent snRNA (Lake
  healthy donors).
- **Injury.** Mouse AKI reduces S2 − S1 amplitude only modestly but S3 − early more. Lake AKI and
  CKD donors have lower amplitude on a human axis than healthy donors. Injury therefore lowers
  zonation amplitude in both species, yet healthy human donors are still flatter than healthy
  mice.

**Consequence for R3.** Most robust pathways are consistent with "mouse zonation, flat human":
with r near 0.15, the human gradient is essentially absent for their genes. Only a few pathways,
and a set of named genes, carry human positional structure beyond that: reversals and genes
zonated in human but not in mouse.

## 10 · Figures

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK, NULL, ACCENT = '#D55E00', '#0072B2', '#333333', '#9A9A9A', '#009E73'


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)


fig, axes = plt.subplots(1, 3, figsize=(15, 6), gridspec_kw={'width_ratios': [1.5, 1, 1]})
ax = axes[0]
forest = main_ratios.copy()
labels_, y_value = [], 0
for contrast in ('S2-S1', 'S3c-early', 'S3-early'):
    block = forest[forest.contrast.eq(contrast)]
    for row in block.itertuples():
        color = HUMAN if row.group == 'human snRNA' else MOUSE
        ax.plot([row.ci_low, row.ci_high], [y_value] * 2, color=color, lw=2)
        ax.scatter(row.ratio_ols, y_value, color=color, s=30, zorder=3)
        if np.isfinite(getattr(row, 'ratio_deming', np.nan)):
            ax.scatter(row.ratio_deming, y_value, marker='D', facecolor='white', edgecolor=color, s=22, zorder=3)
        ax.scatter(row.ratio_orthogonal, y_value, marker='s', facecolor='white', edgecolor=INK, s=18, zorder=3)
        labels_.append(f'{contrast} · {row.reference}')
        y_value -= 1
    y_value -= .6
for row in external_ratios.itertuples():
    ax.plot([row.ci_low, row.ci_high], [y_value] * 2, color=ACCENT, lw=2)
    ax.scatter(row.ratio_ols, y_value, color=ACCENT, s=30, zorder=3)
    labels_.append(f'{row.contrast} · external: {row.analysis.split(",")[0]}')
    y_value -= 1
positions = []
y_value = 0
for contrast in ('S2-S1', 'S3c-early', 'S3-early'):
    n_ = int(forest.contrast.eq(contrast).sum())
    positions += list(np.arange(y_value, y_value - n_, -1))
    y_value -= n_ + .6
positions += list(np.arange(y_value, y_value - len(external_ratios), -1))
ax.set_yticks(positions, labels_, fontsize=6.6)
ax.axvline(FLATTENING, color=NULL, ls='--', lw=.8)
ax.axvline(1, color=INK, lw=.8)
ax.set_xlabel('Human ÷ mouse zonation amplitude (dilution-free slope ratio)')
ax.set_title('a · Amplitude ratio against independent references\n(● OLS with 95% CI · ◇ Deming · □ orthogonal)', loc='left')
ax.text(.99, .01, f'naive slope on our mouse, S2 − S1: {naive["S2-S1"]:.2f}', transform=ax.transAxes, ha='right', fontsize=7)
for k, (name, ref_key) in enumerate((('microdissection male (GSE150338)', 'microdissection male (GSE150338)'),
                                     ('Lake healthy human cortex', 'Lake healthy human cortex'))):
    ax = axes[1 + k]
    ref = references['S2-S1'][ref_key][1]
    genes = ref['gradient'].index.intersection(our_genes['S2-S1'])
    x = ref['gradient'].loc[genes]
    for block, color, label in ((expected['mouse'], MOUSE, 'our mouse'), (expected['human'], HUMAN, 'our human')):
        yv = species_mean(ours, 'S2-S1', block).loc[genes]
        ax.scatter(x, yv, s=3, alpha=.25, color=color, rasterized=True)
        b = ols_slope(x, yv)
        xs = np.linspace(x.quantile(.005), x.quantile(.995), 10)
        ax.plot(xs, yv.mean() + b * (xs - x.mean()), color=color, lw=2, label=f'{label}: slope {b:.2f}')
    ax.axhline(0, color=NULL, lw=.6)
    ax.axvline(0, color=NULL, lw=.6)
    ax.set(xlabel=f'{name}: S2 − S1 (log2)', ylabel='Our S2 − S1 (log2, pseudobulk)')
    ax.legend(frameon=False, fontsize=7.5, loc='upper left')
    ax.set_title(f'{"bc"[k]} · Both species on {name.split(" (")[0]}', loc='left')
fig.tight_layout()
save(fig, 'fig_amplitude_ratio')
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
ax = axes[0]
plot_checks = measurement_checks.copy()
for k, ref_name in enumerate(primary_refs):
    block = plot_checks[plot_checks.reference.eq(ref_name)]
    xs = np.arange(len(block)) + (k - 1) * .22
    color = HUMAN if 'Lake' in ref_name else (MOUSE if 'micro' in ref_name else '#56B4E9')
    ax.errorbar(xs, block.ratio_ols, yerr=[block.ratio_ols - block.ci_low, block.ci_high - block.ratio_ols], fmt='o',
                color=color, ms=4, lw=1.2, label=ref_name.split(' (')[0])
ax.set_xticks(range(len(block)), [a.replace('human expression tertile: ', 'tertile ') for a in block.analysis], rotation=30,
              ha='right', fontsize=7)
ax.axhline(FLATTENING, color=NULL, ls='--', lw=.8)
ax.axhline(1, color=INK, lw=.8)
ax.set_ylabel('Human ÷ mouse ratio (S2 − S1)')
ax.legend(frameon=False, fontsize=6.5)
ax.set_title('d · Measurement checks', loc='left')
ax = axes[1]
compare = pd.DataFrame({'human ÷ mouse (ours)': main_ratios[main_ratios.contrast.eq('S2-S1')].set_index('reference').ratio_ols,
                        'AKI ÷ control (mouse)': aki.ratio_ols,
                        'low-injury halves': low_half.ratio_ols})
compare = compare.loc[[k for k in primary_refs]]
compare.T.plot.bar(ax=ax, color=['#0072B2', '#56B4E9', HUMAN], width=.75, legend=False)
ax.axhline(1, color=INK, lw=.8)
ax.axhline(FLATTENING, color=NULL, ls='--', lw=.8)
ax.set_xticklabels(compare.columns, rotation=20, ha='right', fontsize=7)
ax.set_ylabel('Amplitude ratio (S2 − S1)')
ax.legend([k.split(' (')[0] for k in primary_refs], frameon=False, fontsize=6.5)
ax.set_title('e · Injury does not reproduce the gap', loc='left')
ax = axes[2]
colors = lake_scores.disease.map({'normal': ACCENT, 'acute kidney injury': HUMAN, 'chronic kidney disease': '#7f7f7f'})
ax.scatter(lake_scores.injury, lake_scores.amplitude_S2_S1, c=colors, s=30)
for disease, color in (('normal', ACCENT), ('acute kidney injury', HUMAN), ('chronic kidney disease', '#7f7f7f')):
    ax.scatter([], [], c=color, label=disease)
ax.set(xlabel='Donor injury score (within-Lake z)', ylabel='Donor S2 − S1 amplitude (slope on microdissection)')
ax.legend(frameon=False, fontsize=6.5)
ax.set_title(f'f · Lake donors: ρ = {lake_tests["Spearman injury vs amplitude (all donors)"]:.2f}', loc='left')
ax = axes[3]
lead_plot = lead_table.dropna(subset=['human S2−S1', 'mouse S2−S1'])
klass_colors = {'consistent with flattening': NULL, 'reversal': HUMAN, 'human-specific gradient': '#CC79A7',
                'human-stronger': ACCENT, 'other beyond flattening': INK, 'not testable': '#DDDDDD'}
ax.scatter(lead_plot['mouse S2−S1'].astype(float), lead_plot['human S2−S1'].astype(float),
           c=lead_plot['S2−S1 class'].map(klass_colors), s=22)
for g, row in lead_plot.iterrows():
    if row['S2−S1 class'] != 'consistent with flattening':
        ax.annotate(g, (row['mouse S2−S1'], row['human S2−S1']), fontsize=6.5, xytext=(2, 2), textcoords='offset points')
xs = np.linspace(lead_plot['mouse S2−S1'].astype(float).min(), lead_plot['mouse S2−S1'].astype(float).max(), 10)
ax.plot(xs, r * xs, color=INK, lw=1, ls='--', label=f'flattening: human = {r:.2f} × mouse')
ax.plot(xs, xs, color=NULL, lw=.8, label='equal amplitude')
ax.axhline(0, color='#DDDDDD', lw=.6)
ax.axvline(0, color='#DDDDDD', lw=.6)
ax.set(xlabel='Mouse S2 − S1 (log2)', ylabel='Human S2 − S1 (log2)')
ax.legend(frameon=False, fontsize=6.5, loc='upper left')
ax.set_title('g · Lead genes against global flattening', loc='left')
fig.tight_layout()
save(fig, 'fig_flattening_checks')
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
labels_ph = posthoc_amplitude.index.tolist()
yy = np.arange(len(labels_ph))[::-1]
ax.errorbar(posthoc_amplitude.signal_sd_ratio, yy, xerr=[posthoc_amplitude.signal_sd_ratio - posthoc_amplitude.sd_ratio_ci_low,
            posthoc_amplitude.sd_ratio_ci_high - posthoc_amplitude.signal_sd_ratio], fmt='o', color=INK, ms=5)
ax.set_yticks(yy, labels_ph, fontsize=7)
ax.axvline(1, color=NULL, lw=.8)
ax.set_xlabel('Noise-corrected SD of gradients, first ÷ second group')
ax.set_title('h · Reference-free amplitude (post hoc)', loc='left')
ax = axes[1]
ax.errorbar(posthoc_amplitude.disattenuated_r, yy, xerr=[posthoc_amplitude.disattenuated_r - posthoc_amplitude.r_ci_low,
            posthoc_amplitude.r_ci_high - posthoc_amplitude.disattenuated_r], fmt='o', color=HUMAN, ms=5)
ax.set_yticks(yy, [''] * len(yy))
ax.axvline(1, color=NULL, lw=.8)
ax.axvline(0, color=NULL, lw=.6)
ax.set_xlabel('Disattenuated correlation of gene gradients')
ax.set_title('i · Same genes zonated? (post hoc)', loc='left')
fig.tight_layout()
save(fig, 'fig_reference_free_amplitude')
plt.show()
plt.close(fig)

## 11 · Run record

In [ ]:
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'n_boot': N_BOOT,
            'genes_per_contrast': {c: int(len(g)) for c, g in our_genes.items()},
            'lake_donors': lake_scores.disease.value_counts().to_dict(), 'mouse_snRNA_donors': sn_sex.value_counts().to_dict(),
            'decisions': decisions.astype(str).to_dict(), 'naive_slope': naive,
            'posthoc_reference_free': posthoc_amplitude[['signal_sd_ratio', 'disattenuated_r']].round(3).to_dict(orient='index'),
            'caveat': 'Two mice and two cortex sections from one human donor; segment labels define every contrast; '
                      'reference-based ratios bracket the global ratio (mouse references bias it down, human references up).'}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest['decisions'], indent=2))